# Databricks Vector Search & LangGraph 활용 가이드

---

## 0. 강의 목표

이 노트북은 다음을 목표로 한다.

- Databricks 환경에서 **LLM 기반 서비스**를 어떻게 설계할 수 있는지 이해
- Vector Search Index가 **왜 필요한지**
- LangGraph가 **기존 RAG/Chain 방식보다 왜 유리한지**
- 두 기술을 결합했을 때 **어떤 활용 시나리오가 가능한지**

> 단순한 챗봇이 아니라  
> **사내 데이터를 이해하고 판단하는 AI 구조**를 만드는 것이 목적이다.

---


## 2. Vector Search Index란 무엇인가

#### 2.1 왜 SQL만으로 부족한가?
| 데이터 유형 | SQL | Vector Search |
| :--- | :---: | :---: |
| **정형 데이터** (수치, 날짜 등) | O | △ |
| **문서 / 가이드 / 설명** | X | **O** |
| **정책 / 매뉴얼** | X | **O** |

* 키워드를 정확히 몰라도 의미 기반으로 검색하고 싶은 경우 필수적이다.

#### 2.2 Vector Search 개념
* **임베딩**: 텍스트를 고차원 벡터(숫자 리스트)로 변환.
* **유사도 검색**: 단순 단어 일치가 아닌 의미적 유사도로 검색.
* **작동 원리**: 질문과 가장 비슷한 의미를 가진 문서 조각(Chunk)을 찾아냄.

#### 2.3 Databricks Vector Search 특징
* 원본 데이터와 인덱스가 분리되지 않고 연결됨.
* 거버넌스 및 권한 관리가 중앙에서 이루어짐.
* 원본 데이터 변경 시 인덱스가 실시간 업데이트됨.

---

## 3 LangGraph: 지능형 오케스트레이션
* **정의**: LLM 기반 로직을 **그래프(Graph) 구조**로 설계하는 프레임워크.
* **핵심 개념**:
    * **State**: 전체 프로세스에서 공유되는 정보 (질문, 결과, 판단 근거).
    * **Node**: 특정 역할(검색, SQL 실행, 요약)을 가진 함수 단위.
    * **Edge/Router**: 조건에 따라 다음 노드로 가는 경로 결정.
    * **Graph Builder**:`Node, Edge, State`를 조립해서‘실행 가능한 그래프’를 만드는 역할
---

In [0]:
%pip install langgraph langchain_core
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
!pip list | grep langgraph

#### 3.1 State: 전체 프로세스에서 공유되는 정보
> “지금까지 무슨 일이 있었는가”

In [0]:
from typing import Annotated 
from typing_extensions import TypedDict, List 
from langchain_core.messages import BaseMessage

from langgraph.graph.message import add_messages
from operator import add

class SampleState(TypedDict):

    # 메세지 ([human message, ai message ...])
    messages: Annotated[list[BaseMessage], add_messages]  # LangGraph가 append 머지
    
    # 임의 데이터
    add_data : Annotated[List[str], add]

    # 임의 데이터
    tmp_data : str

    # 기타 임의 데이터
    # ...

---

#### 3.2 Node: 특정 역할을 가진 함수 단위.
>“이 단계에서 무슨 일을 하나”

In [0]:
def first_node(state: SampleState) -> SampleState:
    
    data = ["hello", "world"] 

    return {"add_data": data, "tmp_data": data}
  
def second_node(state: SampleState) -> SampleState:
    
    data = ["HELLO", "WORLD"] 

    return {"add_data": data, "tmp_data": data}

---
#### 3.3 Graph Builder:`Node, Edge, State`를 조립해서‘실행 가능한 그래프’를 만드는 역할
> “이 시스템은 어떻게 흘러가는가”

#### 3.4 Edge/Router: 조건에 따라 다음 노드로 가는 경로 결정.
> “다음엔 어디로 갈까”

In [0]:
from langgraph.graph import StateGraph, START, END

# StateGraph 는 상태(State) 객체의 구조를 정의하는 클래스(또는 타입)를 전달
# LangGraph의 노드들은 실행될 때 이 SampleState를 인자로 받고, 수정된 SampleState를 반환하여 정보를 전달

graph_builder = StateGraph(SampleState)

# 노드 추가
nodes = {
    "first": first_node,
    "second": second_node,
}
for node_name, node_func in nodes.items():
    graph_builder.add_node(node_name, node_func)

# 엣지 추가
graph_builder.add_edge(START, "first")
graph_builder.add_edge("first", "second")
graph_builder.add_edge("second", END)

# 컴파일 
# compile()을 호출하면 이 설계도를 바탕으로 내부적인 유효성 검사를 마치고, 실제 데이터를 넣어 돌릴 수 있는 실행 객체(app)가 생성
app = graph_builder.compile()

In [0]:
from IPython.display import HTML, display

mm = app.get_graph().draw_mermaid()  # Mermaid 소스 문자열

html = f"""
<div class="mermaid">{mm}</div>
<script type="module">
  import mermaid from 'https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.esm.min.mjs';
  mermaid.initialize({{ startOnLoad: true }});
</script>
"""
display(HTML(html))

In [0]:
app.invoke({"messages": [{"role": "user", "content": "안녕?"}]})

{'messages': [HumanMessage(content='안녕?', additional_kwargs={}, response_metadata={}, id='6dbde71a-51e1-4166-94c3-64978a8be021')],
 'add_data': ['hello', 'world', 'HELLO', 'WORLD'],
 'tmp_data': ['HELLO', 'WORLD']}

## 4.LangGraph 패턴으로 이해하기

LangGraph는 기능 단위 라이브러리가 아니라  
**LLM 기반 시스템을 설계하는 패턴 모음**으로 이해하는 것이 좋다.

아래는 실무와 교육에서 가장 많이 사용되는 대표적인 LangGraph 패턴들이다.

#### 4-1. Linear Flow 패턴 (기본 체인)

<table>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      Node가 순차적으로만 실행되는 가장 단순한 구조
    </td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      <img src="/Workspace/Shared/5강_Databricks_AI_Agent_활용/img/Linear Flow 패턴.png" width="800" />
    </td>
  </tr>
</table>

#### 4-2. Router 패턴 (의도 분기)

<table>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      질문 의도에 따라 서로 다른 Node 경로로 분기
    </td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      <img src="/Workspace/Shared/5강_Databricks_AI_Agent_활용/img/Router 패턴.png" width="800" />
    </td>
  </tr>
</table>

#### 4-3. Fan-out / Fan-in 패턴 (병렬 처리)
<table>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      여러 Node를 동시에 실행
    </td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      <img src="/Workspace/Shared/5강_Databricks_AI_Agent_활용/img/Fan-out  Fan-in 패턴.png" width="800" />
    </td>
  </tr>
</table>

#### 4-4. Conditional Loop 패턴 (재시도 / 반복)
<table>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      조건이 만족될 때까지 Node를 반복 실행
    </td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      <img src="/Workspace/Shared/5강_Databricks_AI_Agent_활용/img/Conditional Loop 패턴.png" width="800" />
    </td>
  </tr>
</table>

#### 4-5. Tool-Driven 패턴 (LLM이 Node 선택)
<table>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      LLM이 어떤 Node를 사용할지 결정
    </td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      <img src="/Workspace/Shared/5강_Databricks_AI_Agent_활용/img/Tool-Driven 패턴.png" width="800" />
    </td>
  </tr>
</table>

#### 4-6. Subgraph 패턴 (구조화 / 모듈화)
<table>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      그래프 안에 또 다른 그래프
    </td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      <img src="/Workspace/Shared/5강_Databricks_AI_Agent_활용/img/Subgraph 패턴.png" width="800" />
    </td>
  </tr>
</table>